# 06 — Long-horizon features: seasonal climatology and per-cell trend

Per `notebooks/05_leaderboard_gap_investigation.ipynb`'s conclusion: 78% of Test.csv
sits 10-40 months from the training cutoff, a regime where persistence/neighbourhood
-style signals decay fastest. This notebook tries two techniques aimed specifically at
that regime, gated independently against `evaluate.horizon_matched_split` (this
project's honest proxy metric, RMSE 0.6532 for the current production pipeline):

1. **Seasonal climatology** — each cell's mean `TWS_t` for the same calendar month in
   all *strictly earlier* years, plus a standardised deviation from it. Source for the
   deviation/z-score design: a DrivenData seasonal-streamflow-forecasting competition
   winner write-up (`docs/DrivenData - Seasonal streamflow forecasting winner writeup.pdf` — z-scored
   physical measurements by site and time-of-year), and Hyndman & Athanasopoulos,
   *Forecasting: Principles and Practice* (seasonal-naive/climatology method) for the
   raw climatology mean itself.
2. **Long-window per-cell trend** — a rolling linear-trend slope of `TWS_t` over the
   cell's own trailing history. Source for the leakage-safe rolling-window design
   (excluding the target's own future observations): featuretools' `RollingTrend`
   `gap`/`window_length` pattern (https://featuretools.alteryx.com/en/v1.31.0/guides/time_series.html)
   — implemented here in plain pandas, no new dependency, consistent with the rest of
   `src/features.py`.

Both explored on featuretools' rolling-window *design pattern*, not the library
itself — see project decision log for why (single flat panel table, no relational
schema to exploit; not worth the dependency for two features).

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import config, data, evaluate, features, model
from src.train import get_feature_cols

train, test, sample_submission = data.load_raw_data()
print("train:", train.shape, "| test:", test.shape)


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.5.1 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "C:\Users\alher\anaconda3\Lib\runpy.py", line 198, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "C:\Users\alher\anaconda3\Lib\runpy.py", line 88, in _run_code
    exec(code, run_globals)
  File "C:\Users\alher\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\alher\anaconda3\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\alher\anaconda3\Lib\site-pac

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.5.1 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "C:\Users\alher\anaconda3\Lib\runpy.py", line 198, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "C:\Users\alher\anaconda3\Lib\runpy.py", line 88, in _run_code
    exec(code, run_globals)
  File "C:\Users\alher\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\alher\anaconda3\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\alher\anaconda3\Lib\site-pac

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.5.1 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



train: (2154021, 13) | test: (280961, 13)


## Recompute the current production baseline (honest metric)

Same pipeline as `src/train.py`: masked-`TWS_t` backward fill + spatial-neighbourhood
features, scored on `evaluate.horizon_matched_split`. Recomputed fresh here (not
reused from README) since this is the reference every candidate below is compared
against.

In [2]:
test_filled = features.fill_masked_tws(train, test)

cell_index = (
    pd.concat([train[[config.LAT_COL, config.LON_COL]], test[[config.LAT_COL, config.LON_COL]]])
    .drop_duplicates()
)
cell_id, adjacency, n_cells = features.build_spatial_adjacency(cell_index)
train_nb = features.add_neighbourhood_features(train, cell_id, adjacency, n_cells)
test_nb = features.add_neighbourhood_features(test_filled, cell_id, adjacency, n_cells)

feature_cols = get_feature_cols(train_nb)
print("Base feature cols:", feature_cols)

target_horizons = evaluate.compute_test_horizons(train, test)
fit_df, val_df = evaluate.horizon_matched_split(train_nb, target_horizons)


def score(fit_df, val_df, cols):
    X_fit = features.select_base_features(fit_df, cols)
    y_fit = fit_df[config.TARGET_COL].to_numpy()
    X_val = features.select_base_features(val_df, cols)
    y_val = val_df[config.TARGET_COL].to_numpy()
    m = model.make_baseline_model()
    m.fit(X_fit, y_fit)
    y_pred = model.predict(m, X_val)
    return evaluate.compute_metrics(y_val, y_pred)


baseline_metrics = score(fit_df, val_df, feature_cols)
print("Baseline (honest metric):", baseline_metrics)

Base feature cols: ['TWS_t', 'month_sin', 'month_cos', 'SPEI_01_t', 'SPEI_03_t', 'SPEI_06_t', 'SPEI_12_t', 'SOIL_MOISTURE_t', 'tws_neighbour_mean', 'tws_local_deviation']


C:\Users\alher\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] El sistema no puede encontrar el archivo especificado
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "C:\Users\alher\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "C:\Users\alher\anaconda3\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\alher\anaconda3\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "C:\Users\alher\anaconda3\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _wina

Baseline (honest metric): {'rmse': 0.6532058228124328, 'mae': 0.4520155304549667, 'r2': 0.4252948397012284}


## Candidate 1: seasonal climatology

For each row, the mean (and std) of that cell's own `TWS_t` in the same calendar
month across all *strictly earlier* years. Since a calendar month occurs once per
year, "all earlier years' same month" is automatically `< t` — no leakage, no need
for an explicit gap parameter. Rows with no prior occurrence (a cell's very first
year in the dataset) get `NaN`, left for the pipeline's median imputer.

`tws_climatology_deviation = (TWS_t - climatology_mean) / climatology_std` follows
the DrivenData streamflow winner's z-score-by-location-and-time-of-year pattern —
normalises for each cell's own variability instead of just reporting the raw level.

In [3]:
def add_seasonal_climatology_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    ordered = out.sort_values([config.LAT_COL, config.LON_COL, config.TIME_COL])
    month = ordered[config.TIME_COL].dt.month
    grp = ordered.groupby([ordered[config.LAT_COL], ordered[config.LON_COL], month])[config.TWS_COL]
    clim_mean = grp.transform(lambda s: s.expanding().mean().shift(1))
    clim_std = grp.transform(lambda s: s.expanding().std().shift(1))
    out["tws_climatology_mean"] = clim_mean.reindex(out.index)
    out["tws_climatology_std"] = clim_std.reindex(out.index)
    with np.errstate(invalid="ignore", divide="ignore"):
        deviation = (out[config.TWS_COL] - out["tws_climatology_mean"]) / out["tws_climatology_std"]
    out["tws_climatology_deviation"] = deviation.replace([np.inf, -np.inf], np.nan)
    return out


combined = pd.concat([train_nb, test_nb], ignore_index=True)
combined_clim = add_seasonal_climatology_features(combined)
train_clim = combined_clim.iloc[: len(train_nb)].reset_index(drop=True)
test_clim = combined_clim.iloc[len(train_nb):].reset_index(drop=True)

print(f"Climatology coverage (train): {train_clim['tws_climatology_mean'].notna().mean():.1%}")
print(f"Climatology coverage (test):  {test_clim['tws_climatology_mean'].notna().mean():.1%}")

Climatology coverage (train): 91.3%
Climatology coverage (test):  100.0%


In [4]:
fit_df_c, val_df_c = evaluate.horizon_matched_split(train_clim, target_horizons)
clim_cols = feature_cols + ["tws_climatology_mean", "tws_climatology_deviation"]
clim_metrics = score(fit_df_c, val_df_c, clim_cols)
print("Baseline + seasonal climatology:", clim_metrics)
print("RMSE improvement:", baseline_metrics["rmse"] - clim_metrics["rmse"])

Baseline + seasonal climatology: {'rmse': 0.6505005106485151, 'mae': 0.45946213664795615, 'r2': 0.43004537076560523}
RMSE improvement: 0.0027053121639176725


## Candidate 2: long-window per-cell trend

Rolling linear-trend slope of `TWS_t` over each cell's trailing `window_size`
*observed rows* (not a fixed calendar span — cells with gaps get a window that
covers slightly more than `window_size` months; the slope's own x-axis uses the
real month-index so gaps don't bias its magnitude, only which months are pooled
into it). The window ends at, and includes, the row's own month — legitimate,
since `TWS_t` itself is already an established feature (same principle as
persistence). Computed via the closed-form OLS slope
(`Cov(x,y)/Var(x)`, expanded into rolling first/second moments) rather than
`rolling().apply(np.polyfit)`, to stay vectorised at this row count.

In [5]:
def add_long_window_trend_feature(df: pd.DataFrame, window_size: int = 24) -> pd.DataFrame:
    out = df.copy()
    ordered = out.sort_values([config.LAT_COL, config.LON_COL, config.TIME_COL]).copy()
    ordered["_month_num"] = ordered[config.TIME_COL].dt.year * 12 + ordered[config.TIME_COL].dt.month

    x = ordered["_month_num"].astype(float)
    y = ordered[config.TWS_COL]
    group_keys = [ordered[config.LAT_COL], ordered[config.LON_COL]]
    min_periods = max(3, window_size // 4)

    def roll_mean(s):
        return s.groupby(group_keys).transform(
            lambda v: v.rolling(window_size, min_periods=min_periods).mean()
        )

    mean_x, mean_y = roll_mean(x), roll_mean(y)
    mean_xy, mean_x2 = roll_mean(x * y), roll_mean(x * x)

    var_x = mean_x2 - mean_x ** 2
    cov_xy = mean_xy - mean_x * mean_y
    with np.errstate(invalid="ignore", divide="ignore"):
        slope = (cov_xy / var_x).replace([np.inf, -np.inf], np.nan)

    out["tws_trend_slope"] = slope.reindex(out.index)
    return out


combined_trend = add_long_window_trend_feature(combined)
train_trend = combined_trend.iloc[: len(train_nb)].reset_index(drop=True)
test_trend = combined_trend.iloc[len(train_nb):].reset_index(drop=True)

print(f"Trend coverage (train): {train_trend['tws_trend_slope'].notna().mean():.1%}")
print(f"Trend coverage (test):  {test_trend['tws_trend_slope'].notna().mean():.1%}")

Trend coverage (train): 96.4%
Trend coverage (test):  100.0%


In [6]:
fit_df_t, val_df_t = evaluate.horizon_matched_split(train_trend, target_horizons)
trend_cols = feature_cols + ["tws_trend_slope"]
trend_metrics = score(fit_df_t, val_df_t, trend_cols)
print("Baseline + long-window trend:", trend_metrics)
print("RMSE improvement:", baseline_metrics["rmse"] - trend_metrics["rmse"])

Baseline + long-window trend: {'rmse': 0.653492560151862, 'mae': 0.4527024389395644, 'r2': 0.42479017305063027}
RMSE improvement: -0.00028673733942918833


## Controlled comparison: all variants, plus horizon-bucket breakdown

Same model/fit/split throughout — only the feature columns differ. Horizon buckets
(`< 10` months vs. `>= 10` months from the fit cutoff) match the short/long split
used in `notebooks/05_leaderboard_gap_investigation.ipynb`.

In [7]:
def bucketed_rmse(fit_df, val_df, cols, cutoff):
    X_fit = features.select_base_features(fit_df, cols)
    y_fit = fit_df[config.TARGET_COL].to_numpy()
    m = model.make_baseline_model()
    m.fit(X_fit, y_fit)

    val_df = val_df.copy()
    val_df["horizon"] = evaluate.compute_horizons(cutoff, val_df[config.TIME_COL])
    X_val = features.select_base_features(val_df, cols)
    y_val = val_df[config.TARGET_COL].to_numpy()
    y_pred = model.predict(m, X_val)
    val_df["sq_error"] = (y_val - y_pred) ** 2

    short = val_df[val_df["horizon"] < 10]
    long_ = val_df[val_df["horizon"] >= 10]
    return {
        "short_rmse": float(np.sqrt(short["sq_error"].mean())),
        "long_rmse": float(np.sqrt(long_["sq_error"].mean())),
    }


combined_both = add_long_window_trend_feature(combined_clim)
train_both = combined_both.iloc[: len(train_nb)].reset_index(drop=True)
fit_df_b, val_df_b = evaluate.horizon_matched_split(train_both, target_horizons)
both_cols = feature_cols + ["tws_climatology_mean", "tws_climatology_deviation", "tws_trend_slope"]
both_metrics = score(fit_df_b, val_df_b, both_cols)

cutoff = fit_df[config.TIME_COL].max()
summary = pd.DataFrame({
    "baseline": {**baseline_metrics, **bucketed_rmse(fit_df, val_df, feature_cols, cutoff)},
    "+ climatology": {**clim_metrics, **bucketed_rmse(fit_df_c, val_df_c, clim_cols, cutoff)},
    "+ trend": {**trend_metrics, **bucketed_rmse(fit_df_t, val_df_t, trend_cols, cutoff)},
    "+ climatology + trend": {**both_metrics, **bucketed_rmse(fit_df_b, val_df_b, both_cols, cutoff)},
}).T
summary

,rmse,mae,r2,short_rmse,long_rmse
baseline,0.653206,0.452016,0.425295,0.464429,0.704804
+ climatology,0.650501,0.459462,0.430045,0.472358,0.699703
+ trend,0.653493,0.452702,0.424790,0.463447,0.705373
+ climatology + trend,0.646349,0.458313,0.437296,0.469046,0.695304


## Gate decision

**Mixed result — does not cleanly clear this project's stated gate** ("no
regression on any metric component"). Every climatology-containing variant
improves RMSE (the only metric Zindi actually scores, Phase 1, 50% of the final
score) and R², and specifically helps the targeted long-horizon bucket (`>= 10`
months) — but regresses MAE by ~1.4-1.7%, consistently across every variant
tested (mean-only, deviation-only, combined). Trend alone is a clean loss on
every component; combined with climatology it produces this project's
second-largest single-change RMSE win (1.06%, after masked-fill's 11%), ahead of
the spatial-neighbourhood win (0.8%) — but inherits the same MAE regression.

This project's `horizon_matched_split` proxy already has a known, real gap to the
actual leaderboard (0.6532 estimated vs. 0.7822 actual, see notebook 05) that
isn't fully explained — plausibly 2016-2018 GRACE→GRACE-FO distribution shift
Train.csv can't validate against. Given that gap, and that the RMSE/MAE
disagreement here is a genuine, reproducible trade-off rather than noise, this
project is deferring the graduation decision to **real Zindi submissions**
instead of picking a winner from the internal proxy alone (project decision,
2026-09-04) — three candidate submission files are generated below:
`submission_climatology.csv`, `submission_trend.csv`,
`submission_climatology_trend.csv`. Daily limit is 5 submissions / 200 total
(`docs/rules.txt`), so this uses 3 of today's 5.

**Not graduated to `src/features.py` yet** — pending the real-leaderboard
comparison. Whichever variant(s) win there will be graduated next, with unit
tests, per the usual rule.

## Generate the three candidate submission files

Final full-fit (all of Train.csv, no held-out split) for each candidate feature
set, predicting on `test_filled`/`test_nb` (masked `TWS_t` already backward-filled
and given neighbourhood features, matching `src/train.py`'s production path).

In [8]:
def fit_predict_full(train_df, test_df, cols):
    X_train = features.select_base_features(train_df, cols)
    y_train = train_df[config.TARGET_COL].to_numpy()
    m = model.make_baseline_model()
    m.fit(X_train, y_train)
    X_test = features.select_base_features(test_df, cols)
    return model.predict(m, X_test)


candidates = {
    "submission_climatology.csv": (train_clim, test_clim, clim_cols),
    "submission_trend.csv": (train_trend, test_trend, trend_cols),
    "submission_climatology_trend.csv": (train_both, combined_both.iloc[len(train_nb):].reset_index(drop=True), both_cols),
}

for filename, (tr, te, cols) in candidates.items():
    preds = fit_predict_full(tr, te, cols)
    out_path = config.OUTPUTS_DIR / filename
    data.save_submission(test[config.ID_COL], preds, out_path)
    print(f"Saved {out_path} | pred stats: mean={preds.mean():.3f} std={preds.std():.3f}")

Saved C:\Users\alher\Desktop\Step_Ahead_Drought\outputs\submission_climatology.csv | pred stats: mean=-0.084 std=0.722


Saved C:\Users\alher\Desktop\Step_Ahead_Drought\outputs\submission_trend.csv | pred stats: mean=-0.074 std=0.768


Saved C:\Users\alher\Desktop\Step_Ahead_Drought\outputs\submission_climatology_trend.csv | pred stats: mean=-0.088 std=0.730
